# NB05 — Regime Detection: HMM & Markov Switching

**Objectives**: Fit 2/3-state HMMs, Viterbi decoding, transition matrices,
regime-conditional statistics, model selection via BIC.

**Output**: `regime_labels.parquet`, `hmm_model_params.pkl`, `transition_matrices.csv`

In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.config import *
from src.feature_engineering import compute_log_returns
from src.regime_utils import *
from src.visualization import plot_regime_overlay
print('Imports OK')

Imports OK


## 1. Load Data

In [2]:
master = pd.read_parquet(MASTER_DATA_FILE)
adj_tickers = [t for t in TICKERS if t in master.columns]
log_ret = compute_log_returns(master[adj_tickers])

## 2. Sector-Level HMM (XLK or EW tech)

In [3]:
xlk_col = [c for c in master.columns if 'XLK' in c]
if xlk_col:
    data = compute_log_returns(master[[xlk_col[0]]])[xlk_col[0]].dropna().values
    dates = compute_log_returns(master[[xlk_col[0]]])[xlk_col[0]].dropna().index
else:
    data = log_ret[adj_tickers].mean(axis=1).dropna().values
    dates = log_ret[adj_tickers].mean(axis=1).dropna().index

best_n, bics = select_n_states(data, state_range=[2, 3, 4])
print(f'Best n_states: {best_n}, BICs: {bics}')

Model is not converging.  Current: 7383.665157835032 is not greater than 7383.672030806755. Delta is -0.00687297172316903
Model is not converging.  Current: 7383.723114890301 is not greater than 7383.727415539128. Delta is -0.004300648826756515
Model is not converging.  Current: 7388.096391752732 is not greater than 7388.137328141299. Delta is -0.040936388566478854
Model is not converging.  Current: 7464.202001886113 is not greater than 7464.206933675189. Delta is -0.004931789076181303
Model is not converging.  Current: 7401.339665666993 is not greater than 7401.346239880652. Delta is -0.006574213659405359
Model is not converging.  Current: 7474.77772016412 is not greater than 7474.7940297787445. Delta is -0.016309614624333335
Model is not converging.  Current: 7463.352189905108 is not greater than 7463.394494865456. Delta is -0.042304960347792075
Model is not converging.  Current: 7405.304783570832 is not greater than 7405.305226719584. Delta is -0.00044314875231066253
Model is not co

Best n_states: 3, BICs: {2: np.float64(-14712.961680658404), 3: np.float64(-14818.733639059385), 4: np.float64(-14769.28032645193)}


## 3. Fit & Decode

In [4]:
model, ll = fit_hmm(data, n_states=best_n)
regime_df = decode_regimes(model, data, dates)
regime_df = order_states_by_mean(model, regime_df)
print(regime_df['regime_state'].value_counts().sort_index())

Model is not converging.  Current: 7388.096391752732 is not greater than 7388.137328141299. Delta is -0.040936388566478854
Model is not converging.  Current: 7464.20200188591 is not greater than 7464.206933675458. Delta is -0.004931789547299559
Model is not converging.  Current: 7401.339665666993 is not greater than 7401.346239880652. Delta is -0.006574213659405359


regime_state
0      34
1     886
2    1601
Name: count, dtype: int64


## 4. Transition Matrix

In [ ]:
info = extract_transition_info(model)

# Reorder transition matrix to match relabeled states (0=bear, 1=neutral, 2=bull)
order = np.argsort(model.means_.flatten())
reordered_trans = info['transition_matrix'][order][:, order]
reordered_stat = info['stationary_distribution'][order]
reordered_dur = info['expected_durations'][order]

print('Transition Matrix (reordered: 0=bear, 1=neutral, 2=bull):')
print(pd.DataFrame(reordered_trans,
                   index=['bear', 'neutral', 'bull'],
                   columns=['bear', 'neutral', 'bull']).round(4))
print(f'Stationary dist: {reordered_stat.round(4)}')
print(f'Expected durations: {reordered_dur.round(1)} days')

## 5. Regime-Conditional Statistics

In [ ]:
# Reindex regime labels to match returns index before calling
regime_aligned = regime_df['regime_state'].reindex(log_ret[adj_tickers].dropna().index, method='ffill')
cond_stats = regime_conditional_stats(log_ret[adj_tickers].dropna(), regime_aligned)
cond_stats.head(20)

## 5b. Absorption Ratio as Regime Indicator

Compare the eigenvalue-based Absorption Ratio with HMM regime states.
High AR → tightly coupled markets → systemic fragility. AR spikes should
align with bear/crisis regimes detected by HMM.

In [ ]:
from src.systemic_risk import absorption_ratio

returns_df = log_ret[adj_tickers].dropna()
ar_ts = absorption_ratio(returns_df, k=4, window=252)

# Overlay AR with HMM regime shading
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# Top panel: AR time series
ax1.plot(ar_ts.index, ar_ts.values, linewidth=0.8, color='darkblue')
ax1.axhline(ar_ts.mean(), color='red', linestyle='--', alpha=0.5, label=f'Mean AR: {ar_ts.mean():.3f}')
ax1.set_ylabel('Absorption Ratio')
ax1.set_title('Absorption Ratio vs HMM Regimes')
ax1.legend()

# Shade bear regimes
regime_aligned = regime_df['regime_state'].reindex(ar_ts.index, method='ffill')
bear_mask = regime_aligned == 0  # state 0 = bear (ordered by mean)
for i in range(1, len(bear_mask)):
    if bear_mask.iloc[i]:
        ax1.axvspan(ar_ts.index[i-1], ar_ts.index[i], alpha=0.15, color='red')

# Bottom panel: regime probabilities
for state in sorted(regime_df['regime_state'].unique()):
    prob_col = f'regime_prob_{state}'
    if prob_col in regime_df.columns:
        ax2.plot(regime_df.index, regime_df[prob_col], label=f'P(State {state})', alpha=0.7)
ax2.set_ylabel('Regime Probability')
ax2.legend()

fig.tight_layout()
from src.visualization import save_fig
save_fig(fig, 'nb05_ar_vs_regimes')
plt.show()

# Correlation between AR and bear probability
if 'regime_prob_0' in regime_df.columns:
    aligned = pd.DataFrame({
        'ar': ar_ts, 'p_bear': regime_df['regime_prob_0']
    }).dropna()
    corr = aligned['ar'].corr(aligned['p_bear'])
    print(f"Correlation(AR, P(bear)): {corr:.3f}")
    print("Positive correlation confirms AR rises during bear regimes")

## 6. Save

In [ ]:
regime_df.to_parquet(REGIME_LABELS_FILE)
save_hmm_model(model, 'sector_hmm')
# Save the reordered transition matrix (matching 0=bear, 1=neutral, 2=bull labels)
order = np.argsort(model.means_.flatten())
reordered_trans = info['transition_matrix'][order][:, order]
pd.DataFrame(reordered_trans,
             index=['bear', 'neutral', 'bull'],
             columns=['bear', 'neutral', 'bull']).to_csv(TRANSITION_MATRIX_FILE)
print('Saved all NB05 outputs')